In [1]:
import getpass
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY

In [4]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [5]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vector_1 = embeddings.embed_query(documents[0].page_content)
vector_2 = embeddings.embed_query(documents[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[0.017457252, 0.0045292513, -0.19743718, -0.074493006, 0.09089061, 0.036798697, -0.08895603, 0.029206252, -0.036273994, -0.026525032]


In [6]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embeddings)

In [7]:
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "nke-10k-2023.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))

107


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

516


In [9]:
ids = vector_store.add_documents(documents=all_splits)

In [10]:
print(len(vector_store.store))

516


In [11]:
import numpy

In [12]:
results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

page_content='direct to consumer operations sell products through the following number of retail stores in the United States:
U.S. RETAIL STORES NUMBER
NIKE Brand factory stores 213 
NIKE Brand in-line stores (including employee-only stores) 74 
Converse stores (including factory stores) 82 
TOTAL 369 
In the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for further information.
2023 FORM 10-K 2' metadata={'source': 'nke-10k-2023.pdf', 'page': 4, 'start_index': 3125}


In [13]:
results = await vector_store.asimilarity_search("When was Nike incorporated?")

print(results[0])

page_content='transition of NIKE Brand businesses in certain countries within APLA to third-party distributors.
The Company's NIKE Direct operations are managed within each NIKE Brand geographic operating segment. Converse is also a reportable segment for the Company
and operates in one industry: the design, marketing, licensing and selling of athletic lifestyle sneakers, apparel and accessories.
Global Brand Divisions is included within the NIKE Brand for presentation purposes to align with the way management views the Company. Global Brand Divisions
revenues include NIKE Brand licensing and other miscellaneous revenues that are not part of a geographic operating segment. Global Brand Divisions costs represent
demand creation and operating overhead expense that include product creation and design expenses centrally managed for the NIKE Brand, as well as costs associated
with NIKE Direct global digital operations and enterprise technology.
(1)
2023 FORM 10-K 84' metadata={'source': 'nk

In [14]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)


Score: 0.8398043512170722

page_content='Table of Contents
FISCAL 2023 NIKE BRAND REVENUE HIGHLIGHTSThe following tables present NIKE Brand revenues disaggregated by reportable operating segment, distribution channel and major product line:
FISCAL 2023 COMPARED TO FISCAL 2022
• NIKE, Inc. Revenues were $51.2 billion in fiscal 2023, which increased 10% and 16% compared to fiscal 2022 on a reported and currency-neutral basis, respectively.
The increase was due to higher revenues in North America, Europe, Middle East & Africa ("EMEA"), APLA and Greater China, which contributed approximately 7, 6,
2 and 1 percentage points to NIKE, Inc. Revenues, respectively.
• NIKE Brand revenues, which represented over 90% of NIKE, Inc. Revenues, increased 10% and 16% on a reported and currency-neutral basis, respectively. This
increase was primarily due to higher revenues in Men's, the Jordan Brand, Women's and Kids' which grew 17%, 35%,11% and 10%, respectively, on a wholesale
equivalent basis.' metad

In [15]:
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='Enterprise Resource Planning Platform, data and analytics, demand sensing, insight gathering, and other areas to create an end-to-end technology foundation, which we
believe will further accelerate our digital transformation. We believe this unified approach will accelerate growth and unlock more efficiency for our business, while driving
speed and responsiveness as we serve consumers globally.
FINANCIAL HIGHLIGHTS
• In fiscal 2023, NIKE, Inc. achieved record Revenues of $51.2 billion, which increased 10% and 16% on a reported and currency-neutral basis, respectively
• NIKE Direct revenues grew 14% from $18.7 billion in fiscal 2022 to $21.3 billion in fiscal 2023, and represented approximately 44% of total NIKE Brand revenues for
fiscal 2023
• Gross margin for the fiscal year decreased 250 basis points to 43.5% primarily driven by higher product costs, higher markdowns and unfavorable changes in foreign
currency exchange rates, partially offset by strategic pricing action

In [16]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)


[[Document(id='752f4464-af24-4c8b-a863-6fb095442240', metadata={'source': 'nke-10k-2023.pdf', 'page': 4, 'start_index': 3125}, page_content='direct to consumer operations sell products through the following number of retail stores in the United States:\nU.S. RETAIL STORES NUMBER\nNIKE Brand factory stores 213 \nNIKE Brand in-line stores (including employee-only stores) 74 \nConverse stores (including factory stores) 82 \nTOTAL 369 \nIn the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for further information.\n2023 FORM 10-K 2')],
 [Document(id='5dd43af6-f41f-4cbc-90cd-400a0fd27ce5', metadata={'source': 'nke-10k-2023.pdf', 'page': 86, 'start_index': 3033}, page_content="transition of NIKE Brand businesses in certain countries within APLA to third-party distributors.\nThe Company's NIKE Direct operations are managed within each NIKE Brand geographic operating segment. Converse is also a reportable segment for the Company\nand operates in one

In [17]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

[[Document(id='752f4464-af24-4c8b-a863-6fb095442240', metadata={'source': 'nke-10k-2023.pdf', 'page': 4, 'start_index': 3125}, page_content='direct to consumer operations sell products through the following number of retail stores in the United States:\nU.S. RETAIL STORES NUMBER\nNIKE Brand factory stores 213 \nNIKE Brand in-line stores (including employee-only stores) 74 \nConverse stores (including factory stores) 82 \nTOTAL 369 \nIn the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for further information.\n2023 FORM 10-K 2')],
 [Document(id='5dd43af6-f41f-4cbc-90cd-400a0fd27ce5', metadata={'source': 'nke-10k-2023.pdf', 'page': 86, 'start_index': 3033}, page_content="transition of NIKE Brand businesses in certain countries within APLA to third-party distributors.\nThe Company's NIKE Direct operations are managed within each NIKE Brand geographic operating segment. Converse is also a reportable segment for the Company\nand operates in one

In [33]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Wrap your existing retriever as a tool the agent can choose to call
@tool
def search_nike_10k(query: str) -> str:
    """Search Nike's 2023 10-K filing for relevant information."""
    docs = vector_store.similarity_search(query, k=2)
    return "\n\n".join(d.page_content for d in docs)

llm = llm = ChatOllama(
    model="qwen2.5:7b-instruct",
    temperature=0.3,
)

agent = create_react_agent(llm, tools=[search_nike_10k])

response = agent.invoke({
    "messages": [{"role": "user", "content": "How many distribution centers does Nike have in the US?"}]
})
print(response["messages"][-1].content)

According to Nike's 2023 10-K filing, Nike has eight significant distribution centers in the United States. Five of these are located in or near Memphis, Tennessee, with three being leased and two owned. Two other distribution centers are leased and operated by third-party logistics providers, one in Indianapolis, Indiana, and one in Dayton, Tennessee. Additionally, there is one distribution center for Converse located in Ontario, California, which is also leased.


In [34]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the capital of France"}]
})

print(response["messages"][-1].content)

The capital of France is Paris. If you need more information about Paris or any other details, feel free to ask!
